# Hiver Spotify Agent — Full A100 Pipeline Run

**Runtime**: A100 GPU (40 GB)  
**Estimated total time**: ~45 minutes  
**What this does**:
1. Mounts Google Drive (auto-saves everything there)
2. Clones the repo from GitHub
3. Installs all dependencies
4. Downloads real Spotify Twitter data from Kaggle
5. Trains the intent classifier on real data
6. Builds the FAISS RAG index
7. Runs the full evaluation harness (LLM-as-judge via Groq)
8. Fine-tunes DistilBERT on the golden eval set
9. Saves all results back to Drive

---
> **Before you start**: Make sure your runtime is set to A100.  
> Runtime → Change runtime type → Hardware accelerator → A100

## Step 0 — Anti-Disconnect & Auto-Save

Run this cell first. It keeps the Colab tab alive and auto-saves results to Drive every 5 minutes.

In [ ]:
# ── Anti-disconnect: click into the page output area, then run this ──────────
# Paste this in the browser console (F12 → Console) to prevent idle disconnect:
# function KeepAlive() { document.querySelector('#ok').click() }
# setInterval(KeepAlive, 60000)
#
# The JS trick works in Chrome. As a Python backup, we also touch a heartbeat
# file every 60s in the background so the kernel stays warm.

import threading, time, os

def _heartbeat():
    while True:
        open('/tmp/heartbeat', 'w').close()
        time.sleep(60)

hb = threading.Thread(target=_heartbeat, daemon=True)
hb.start()
print('Heartbeat thread started — kernel will stay warm.')

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = '/content/drive/MyDrive/hiver-spotify-agent-results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Results will be saved to: {DRIVE_DIR}')

In [ ]:
# Auto-save helper — call save_to_drive() after any step, or it runs every 5 min
import shutil, datetime

REPO_DIR = '/content/hiver-spotify-agent'

def save_to_drive(label='checkpoint'):
    ts = datetime.datetime.now().strftime('%H%M%S')
    src = f'{REPO_DIR}/results'
    dst = f'{DRIVE_DIR}/results_{label}_{ts}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved to Drive: {dst}')
    # Also save models
    src_m = f'{REPO_DIR}/models'
    dst_m = f'{DRIVE_DIR}/models'
    if os.path.exists(src_m):
        shutil.copytree(src_m, dst_m, dirs_exist_ok=True)
    # Save data
    src_d = f'{REPO_DIR}/data'
    dst_d = f'{DRIVE_DIR}/data'
    if os.path.exists(src_d):
        shutil.copytree(src_d, dst_d, dirs_exist_ok=True)

def _auto_save_loop():
    while True:
        time.sleep(300)  # every 5 minutes
        try:
            save_to_drive('auto')
        except Exception as e:
            print(f'Auto-save error (non-fatal): {e}')

as_thread = threading.Thread(target=_auto_save_loop, daemon=True)
as_thread.start()
print('Auto-save to Drive running every 5 minutes.')

## Step 1 — Clone repo & verify GPU

In [ ]:
import subprocess, sys

!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

# Clone or pull
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Aprameya05/hiver-spotify-agent.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!ls

## Step 2 — Install dependencies

In [ ]:
# Colab already has torch/transformers; install the rest quietly
!pip install -q groq sentence-transformers faiss-cpu kaggle textblob rich typer python-dotenv
!pip install -q 'scikit-learn>=1.4.0' 'datasets>=2.19.0'
print('Dependencies installed.')

## Step 3 — Set API keys

Fill in your keys below. They are only in memory — never saved to disk.

In [ ]:
import os

# ── FILL IN YOUR KEYS HERE ────────────────────────────────────────────────────
os.environ['GROQ_API_KEY']      = 'YOUR_GROQ_API_KEY_HERE'
os.environ['KAGGLE_USERNAME']   = 'YOUR_KAGGLE_USERNAME_HERE'
os.environ['KAGGLE_KEY']        = 'YOUR_KAGGLE_KEY_HERE'
# ─────────────────────────────────────────────────────────────────────────────

# Write kaggle.json so the CLI works
import json, pathlib
kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(
    json.dumps({'username': os.environ['KAGGLE_USERNAME'],
                'key':      os.environ['KAGGLE_KEY']})
)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Quick sanity check
from groq import Groq
c = Groq(api_key=os.environ['GROQ_API_KEY'])
r = c.chat.completions.create(model='llama-3.3-70b-versatile',
                               messages=[{'role':'user','content':'say hi'}],
                               max_tokens=5)
print('Groq OK:', r.choices[0].message.content.strip())

## Step 4 — Download Kaggle dataset & preprocess

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

!kaggle datasets download -d thoughtvector/customer-support-on-twitter -p data/raw --unzip -q
!ls data/raw/

In [ ]:
# Filter to Spotify threads
import pandas as pd
from src.utils import clean_tweet

df = pd.read_csv('data/raw/twcs.csv', dtype=str)
print('Total rows:', len(df))

# Find Spotify author IDs
spotify_ids = df[df['author_id'].str.lower().str.contains('spotify', na=False)]['author_id'].unique()
print('Spotify author IDs found:', spotify_ids[:10])

# Build threads: customer tweet + spotify reply
spotify_replies = df[df['author_id'].isin(spotify_ids)].copy()
print('Spotify reply rows:', len(spotify_replies))

# Merge to get customer tweet text
customer_tweets = df[~df['author_id'].isin(spotify_ids)][['tweet_id', 'text', 'author_id']].copy()
customer_tweets.columns = ['inbound_tweet_id', 'customer_text', 'customer_author']

merged = spotify_replies.merge(
    customer_tweets,
    left_on='in_response_to_tweet_id',
    right_on='inbound_tweet_id',
    how='inner'
)
print('Matched threads:', len(merged))

# Clean
merged['customer_text_clean'] = merged['customer_text'].apply(clean_tweet)
merged['spotify_reply_clean'] = merged['text'].apply(clean_tweet)

# Drop empties
merged = merged[
    (merged['customer_text_clean'].str.len() > 5) &
    (merged['spotify_reply_clean'].str.len() > 5)
].reset_index(drop=True)
print('After cleaning:', len(merged))

merged.to_parquet('data/processed/spotify_threads.parquet', index=False)
print('Saved to data/processed/spotify_threads.parquet')
merged.head(3)

In [ ]:
save_to_drive('after_preprocess')

## Step 5 — Train intent classifier on real data

In [ ]:
from src.intent_classifier import IntentClassifier

clf = IntentClassifier()
clf.fit_from_seeds()   # warm start with seeds
clf.save()
print('Classifier trained and saved.')

## Step 6 — Build golden eval set (LLM-verified labels)

In [ ]:
# Sample 250 threads, classify, then LLM-verify labels
import json, random
from scripts.build_golden_eval import build_golden_set, extract_threads_from_df

threads = extract_threads_from_df(merged)
random.seed(42)
sample = random.sample(threads, min(250, len(threads)))

golden = build_golden_set(sample, clf, verify_with_llm=True, verbose=True)

os.makedirs('data/golden', exist_ok=True)
with open('data/golden/eval_set.json', 'w') as f:
    json.dump(golden, f, indent=2)
print(f'Golden eval set: {len(golden)} examples saved.')
save_to_drive('after_golden')

## Step 7 — Build FAISS RAG index

In [ ]:
from src.rag_retriever import RAGRetriever

rag = RAGRetriever()
rag.build_index_from_df(merged, customer_col='customer_text_clean', reply_col='spotify_reply_clean')
rag.save()
print('RAG index built and saved.')
save_to_drive('after_rag')

## Step 8 — Run full evaluation pipeline

In [ ]:
import subprocess, time
start = time.time()

result = subprocess.run(
    ['python', '-m', 'scripts.run_pipeline', '--golden', 'data/golden/eval_set.json', '--output', 'results/'],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(result.stdout[-3000:])  # last 3k chars
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

elapsed = time.time() - start
print(f'Eval done in {elapsed/60:.1f} min')
save_to_drive('after_eval')

In [ ]:
# Pretty-print the summary
import json
with open('results/eval_summary.json') as f:
    summary = json.load(f)

import pprint
pprint.pprint(summary)

## Step 9 — Fine-tune DistilBERT on golden set (A100 muscle)

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
)
from src.intent_classifier import INTENT_LABELS
from sklearn.metrics import f1_score, classification_report
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# Load golden set
with open('data/golden/eval_set.json') as f:
    golden = json.load(f)

label2id = {l: i for i, l in enumerate(INTENT_LABELS)}
id2label = {i: l for l, i in label2id.items()}

class IntentDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            ex['text'],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(label2id[ex['intent']], dtype=torch.long),
        }

# 80/20 train/val split
import random
random.seed(42)
random.shuffle(golden)
split = int(len(golden) * 0.8)
train_ex, val_ex = golden[:split], golden[split:]
print(f'Train: {len(train_ex)}, Val: {len(val_ex)}')

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(INTENT_LABELS),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

train_loader = DataLoader(IntentDataset(train_ex, tokenizer), batch_size=16, shuffle=True)
val_loader   = DataLoader(IntentDataset(val_ex,   tokenizer), batch_size=32)

EPOCHS = 8
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=len(train_loader),
    num_training_steps=EPOCHS * len(train_loader)
)

best_f1 = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            preds.extend(logits.argmax(-1).cpu().tolist())
            trues.extend(batch['labels'].cpu().tolist())

    f1 = f1_score(trues, preds, average='weighted')
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch}/{EPOCHS} | loss={avg_loss:.4f} | val F1={f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained('models/distilbert-intent')
        tokenizer.save_pretrained('models/distilbert-intent')
        print(f'  -> New best F1={f1:.4f}, model saved.')

print('\nFinal classification report:')
print(classification_report(trues, preds, target_names=INTENT_LABELS))
save_to_drive('after_finetune')

## Step 10 — Update results/eval_summary.json with real numbers

In [ ]:
import json, os
from sklearn.metrics import f1_score

# Patch the eval summary with the real DistilBERT F1
summary_path = 'results/eval_summary.json'
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
else:
    summary = {}

summary['distilbert_val_f1'] = round(best_f1, 4)
summary['n_golden_examples'] = len(golden)
summary['n_training_examples'] = len(train_ex)
summary['fine_tuning_epochs'] = EPOCHS

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
save_to_drive('final')

## Step 11 — Push results back to GitHub

In [ ]:
GITHUB_PAT = 'YOUR_GITHUB_PAT_HERE'  # paste your PAT here — never commit this

!git config user.email 'aprameya.bharadwaj.05@gmail.com'
!git config user.name 'Aprameya Bharadwaj'
!git remote set-url origin https://{GITHUB_PAT}@github.com/Aprameya05/hiver-spotify-agent.git

!git add results/ models/ data/golden/
!git commit -m "feat: add real pipeline results from A100 run"
!git push origin main
print('Pushed to GitHub.')

## Done!

All results are:
- Saved to your **Google Drive** under `hiver-spotify-agent-results/`
- Pushed to **GitHub** at `Aprameya05/hiver-spotify-agent`

Key numbers to note from `results/eval_summary.json`:
- Intent classifier weighted F1 (seeds + logistic regression)
- DistilBERT fine-tuned val F1
- LLM-as-judge scores (relevance, accuracy, tone, conciseness, actionability)
- Escalation engine precision / recall